<a href="https://colab.research.google.com/github/sooyeon904/kfa-ai-agent/blob/main/w02_%EB%82%B4%EC%86%8C%EC%8A%A4%EC%88%98%EC%A7%91.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KFA AI 에이전트 스터디 사전과제

## 리포트 데이터 소스 찾기

### 선정 리포트

- 증권사: 하나증권
- 리포트 제목: 증권 - 3Q26 Pre, 실적보다는 배당에 초점을
- 발행일: 2026년 10월 14일
- 리포트 링크: [하나증권 증권 산업 리포트 보기](https://stock.naver.com/research/industry/46469)

### 선정 도표

1. 도표 3: 삼성증권 2026년 3분기 연결 실적 예상치
2. 도표 5: 키움증권 2026년 3분기 연결 실적 예상치

두 도표에는 실제 실적과 하나증권의 예상치가 함께 포함되어 있다.
이 과제에서는 실제 실적 중 2026년 2분기 연결 영업이익과 지배주주순이익의 원천자료를 확인하고, Open DART API를 통한 수집을 시도한다.

3분기와 4분기 예상치는 하나증권의 추정치로, DART에서 수집하는 실제 공시 실적과 구분한다.

In [ ]:
from google.colab import userdata

try:
    dart_key = userdata.get("DART_API_KEY")
    if dart_key:
        print("DART 키 확인 완료!")
    else:
        print("DART 키가 비어 있어요.")
except Exception:
    print("DART 키를 불러오지 못했어요.")
    print("왼쪽 열쇠 아이콘에서 키 이름과 노트북 액세스를 확인해 주세요.")

DART 키 확인 완료!


In [ ]:
import requests

# DART에서 사용하는 회사 고유번호
회사목록 = {
    "삼성증권": "00104856",
    "키움증권": "00296290"
}

주소 = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

for 회사명, 고유번호 in 회사목록.items():
    print(f"\n[{회사명}] 조회 중...")

    try:
        응답 = requests.get(
            주소,
            params={
                "crtfc_key": dart_key,
                "corp_code": 고유번호,
                "bsns_year": "2026",
                "reprt_code": "11012",  # 반기보고서
                "fs_div": "CFS"         # 연결 재무제표
            },
            timeout=(10, 20)
        )

        데이터 = 응답.json()

        if 데이터.get("status") == "000":
            print("조회 성공!")
            print("재무제표 항목 수:", len(데이터.get("list", [])))
        else:
            print("조회 결과:", 데이터.get("status"), 데이터.get("message"))

    except requests.exceptions.RequestException:
        print("DART 연결이 지연되거나 실패했어요.")
    except ValueError:
        print("DART의 응답을 읽지 못했어요.")


[삼성증권] 조회 중...
조회 성공!
재무제표 항목 수: 192

[키움증권] 조회 중...
조회 성공!
재무제표 항목 수: 309


In [ ]:
# 두 회사의 연결 포괄손익계산서에서 필요한 항목을 찾습니다.
for 회사명, 고유번호 in 회사목록.items():
    print(f"\n[{회사명}]")

    try:
        응답 = requests.get(
            주소,
            params={
                "crtfc_key": dart_key,
                "corp_code": 고유번호,
                "bsns_year": "2026",
                "reprt_code": "11012",
                "fs_div": "CFS"
            },
            timeout=(10, 20)
        )

        데이터 = 응답.json()

        if 데이터.get("status") != "000":
            print("조회 결과:", 데이터.get("message"))
            continue

        찾은항목 = False

        for 항목 in 데이터.get("list", []):
            이름 = 항목.get("account_nm", "")
            재무제표 = 항목.get("sj_div", "")

            if 재무제표 in ["IS", "CIS"] and (
                이름 == "영업이익"
                or ("지배기업" in 이름 and "순이익" in 이름)
            ):
                찾은항목 = True
                print("항목:", 이름)
                print("  당기 금액:", 항목.get("thstrm_amount", "없음"))
                print("  당기 누적 금액:", 항목.get("thstrm_add_amount", "없음"))

        if not 찾은항목:
            print("조건에 맞는 항목을 찾지 못했어요. 항목 이름을 다시 확인해야 해요.")

    except requests.exceptions.RequestException:
        print("DART 연결이 지연되거나 실패했어요.")
    except ValueError:
        print("DART의 응답을 읽지 못했어요.")


[삼성증권]
항목: 지배기업소유주지분순이익
  당기 금액: 488224283447
  당기 누적 금액: 939080352543
항목: 영업이익
  당기 금액: 675794814040
  당기 누적 금액: 1285326557425

[키움증권]
항목: 지배기업의 소유주에게 귀속되는 반기순이익(손실)
  당기 금액: 680072348384
  당기 누적 금액: 1156435569844
항목: 영업이익
  당기 금액: 788949832477
  당기 누적 금액: 1410170449849


## 데이터 원천 출처 정리

| 차트·표 내용 | 원천 출처 | 링크 | 기준 날짜 | 갱신 주기 | 무료/유료 | 수집 방법 |
|---|---|---|---|---|---|---|
| 도표 3: 삼성증권 연결 실적 중 영업이익·지배주주순이익 | 삼성증권 반기보고서, 연결 포괄손익계산서 | [DART 반기보고서](https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260814002836) | 2026년 2분기 | 분기별 | 무료 | ① Open DART 공식 API |
| 도표 5: 키움증권 연결 실적 중 영업이익·지배주주순이익 | 키움증권 반기보고서, 연결 포괄손익계산서 | [DART 반기보고서](https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260813001767) | 2026년 2분기 | 분기별 | 무료 | ① Open DART 공식 API |

### 확인 결과

각 회사의 2026년 2분기 연결 영업이익과 지배주주순이익을
반기보고서 및 Open DART API 조회 결과와 대조하였다.

리포트의 3분기·4분기 예상치는 하나증권의 추정치이며,
위 표에서 조회한 실제 공시 실적과 구분한다.

## 검증 범위

리서치의 도표 3과 도표 5에 표시된 2026년 2분기 실제 실적 중
연결 영업이익과 지배주주순이익을 확인하였다.

각 회사의 반기보고서와 Open DART API 조회 결과를 대조한 결과,
아래 네 항목이 리서치에 표시된 수치와 일치했다.

| 회사 | 항목 | DART 조회 금액 | 리서치 수치 | 결과 |
|---|---|---:|---:|---|
| 삼성증권 | 연결 영업이익 | 675,794,814,040원 | 675.8십억 원 | 일치 |
| 삼성증권 | 지배주주순이익 | 488,224,283,447원 | 488.2십억 원 | 일치 |
| 키움증권 | 연결 영업이익 | 788,949,832,477원 | 788.9십억 원 | 일치 |
| 키움증권 | 지배주주순이익 | 680,072,348,384원 | 680.1십억 원 | 일치 |

리서치는 십억 원 단위로 소수점 첫째 자리까지 표시한다.
API의 당기 금액은 2분기 3개월 실적이며, 당기 누적 금액은
1~2분기 상반기 누적 실적이다.

도표의 모든 수치와 예상치를 검증한 것은 아니다.
3분기와 4분기 예상치는 하나증권의 추정치로, 공시된 실제 실적과 구분한다.

## 데이터 수집 방법

두 도표에서 확인한 실제 실적은 Open DART 공식 API를 통해 수집했다.

- API: Open DART 단일회사 전체 재무제표
- 수집 대상: 삼성증권, 키움증권
- 사업연도: 2026년
- 보고서: 반기보고서
- 재무제표: 연결재무제표
- 확인 항목: 영업이익, 지배주주순이익
- 기준 기간: 2026년 2분기 3개월 및 상반기 누적
- 단위: 원

### 수집 순서

1. 각 회사의 DART 고유번호를 확인한다.
2. 사업연도, 보고서 종류, 연결재무제표를 지정해 API를 호출한다.
3. 반환된 항목 중 영업이익과 지배주주순이익을 찾는다.
4. 2분기 금액을 반기보고서 및 리서치의 실제 실적과 비교한다.

※ 리서치의 예상치는 하나증권의 추정치로, DART 공시 실적과 구분한다.

In [ ]:
import requests
import pandas as pd
from google.colab import userdata

# 코랩에 저장된 DART 키 사용
dart_key = userdata.get("DART_API_KEY")

# 조사할 회사와 보고서
회사목록 = {
    "삼성증권": "00104856",
    "키움증권": "00296290"
}

기간목록 = [
    ("2025년 3분기", "2025", "11014"),
    ("2026년 2분기", "2026", "11012")
]

주소 = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
결과 = []

for 회사명, 고유번호 in 회사목록.items():
    for 기간, 연도, 보고서코드 in 기간목록:
        print(f"{회사명} {기간} 조회 중...")

        try:
            응답 = requests.get(
                주소,
                params={
                    "crtfc_key": dart_key,
                    "corp_code": 고유번호,
                    "bsns_year": 연도,
                    "reprt_code": 보고서코드,
                    "fs_div": "CFS"
                },
                timeout=(10, 20)
            )

            데이터 = 응답.json()

            if 데이터.get("status") != "000":
                print("조회 실패:", 데이터.get("message"))
                continue

            for 항목 in 데이터.get("list", []):
                이름 = 항목.get("account_nm", "")
                재무제표 = 항목.get("sj_div", "")

                if 재무제표 in ["IS", "CIS"] and (
                    이름 == "영업이익"
                    or ("지배기업" in 이름 and "순이익" in 이름)
                ):
                    결과.append({
                        "회사": 회사명,
                        "기간": 기간,
                        "항목": 이름,
                        "해당 분기 금액(원)": 항목.get("thstrm_amount", ""),
                        "누적 금액(원)": 항목.get("thstrm_add_amount", "")
                    })

        except (requests.exceptions.RequestException, ValueError):
            print("조회 중 연결 또는 응답 오류가 발생했어요.")

if 결과:
    print("\n조회 결과")
    display(pd.DataFrame(결과))
else:
    print("\n조회된 결과가 없어요. 오류 메시지를 확인해 주세요.")

삼성증권 2025년 3분기 조회 중...
삼성증권 2026년 2분기 조회 중...
키움증권 2025년 3분기 조회 중...
키움증권 2026년 2분기 조회 중...

조회 결과


,회사,기간,항목,해당 분기 금액(원),누적 금액(원)
0,삼성증권,2025년 3분기,지배기업소유주지분순이익,309174366337,792231898731
1,삼성증권,2025년 3분기,영업이익,401812408978,1045106742027
2,삼성증권,2026년 2분기,지배기업소유주지분순이익,488224283447,939080352543
3,삼성증권,2026년 2분기,영업이익,675794814040,1285326557425
4,키움증권,2025년 3분기,영업이익,408881346062,1142633564981
5,키움증권,2026년 2분기,지배기업의 소유주에게 귀속되는 반기순이익(손실),680072348384,1156435569844
6,키움증권,2026년 2분기,영업이익,788949832477,1410170449849


In [ ]:
# 키움증권 2025년 3분기의 이익 관련 항목 이름 확인
응답 = requests.get(
    주소,
    params={
        "crtfc_key": dart_key,
        "corp_code": "00296290",
        "bsns_year": "2025",
        "reprt_code": "11014",
        "fs_div": "CFS"
    },
    timeout=(10, 20)
)

데이터 = 응답.json()

if 데이터.get("status") == "000":
    for 항목 in 데이터.get("list", []):
        이름 = 항목.get("account_nm", "")
        재무제표 = 항목.get("sj_div", "")

        if 재무제표 in ["IS", "CIS"] and (
            "순이익" in 이름 or "지배" in 이름
        ):
            print(
                이름,
                "| 해당 분기:", 항목.get("thstrm_amount", ""),
                "| 누적:", 항목.get("thstrm_add_amount", "")
            )
else:
    print("조회 결과:", 데이터.get("status"), 데이터.get("message"))

비지배주주지분 | 해당 분기: 475679726 | 누적: 1145842087
지배주주지분 | 해당 분기: 400184585389 | 누적: 1075716528767
당기순이익 | 해당 분기: 322382473016 | 누적: 868115199969
비지배주주지분 | 해당 분기: 472386850 | 누적: 1164679052
지배주주지분 | 해당 분기: 321910086166 | 누적: 866950520917
법인세비용차감전순이익 | 해당 분기: 439213389866 | 누적: 1181716630653


In [1]:
#@title 🧰 공통 도구 (고치지 말고 실행만. 코드가 궁금하면 이 셀을 더블클릭) { display-mode: "form" }
"""KFA AI agent study: shared collection toolkit (2nd session template).

Every source, whatever its collection level (1 API, 2 library, 3 file,
4 crawling, 5 manual), ends in the same record table via finalize().
Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 목차] 이 셀은 고치지 않고 실행만 해요. 아래는 어떤 함수가 어디에 쓰이는지예요.

  공통 출구 (모든 단계가 마지막에 거치는 곳)
    collect_table()     받아온 표 → 공통 표 15칸. 내 소스 함수의 마지막 줄은 항상 이것
    finalize()          꼬리표 붙이기·숫자/날짜 정리·형식 통과/확인 필요 판정 (collect_table이 불러 씀)
    long_from_table()   칸마다 항목이 하나씩인 넓은 표 → 숫자 하나 = 한 줄 (collect_table이 불러 씀)
    check_records()     공통 표 모양이 맞는지 검사하고 요약 출력
    save_records()      CSV 두 개로 저장 (전부 / 확인 필요만)
    run_step()          예시 하나를 실행하고 ✅ ⏭ ❌ 요약표에 한 줄 남김 (실패해도 멈추지 않음)
    step_summary()      요약표 보여주기

  1단계 공식 API      read_json_table() · collect_dart() · collect_ecos() · get_key()
  2단계 라이브러리     (도구 없음: 라이브러리가 준 표를 collect_table에 바로)
  3단계 파일          read_table()
  4단계 크롤링        read_html_table() · robots_allowed() · find_col()
  5단계 수동 입력      (pd.DataFrame으로 직접) · mount_drive() · read_source_cards() · pick_sheet_title() · collect_from_cards()

  정리 도구           to_number() 숫자 바꾸기 · to_date() 날짜 바꾸기 · is_blank() 빈칸 판단
  약속된 값           NOT_APPLICABLE = "해당 없음" · SAME_AS_DATE = 공시일을 기준일과 같게
  멈춤 신호           SkipStep = 준비 부족이라 건너뜀(⏭) · CollectError = 실패(❌), 이유를 쉬운 말로
"""

import io
import json
import math
import numbers
import os
import re
import urllib.robotparser
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

COLUMNS = [
    "종목코드", "종목명", "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각", "상태", "비고",
]
REQUIRED = [
    "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각",
]
NOT_APPLICABLE = "해당 없음"
NA_ALLOWED = {"종목코드", "종목명", "기준", "공시일"}
RELIABILITY = ("상", "중", "하")
LEVELS = {1: "공식 API", 2: "파이썬 라이브러리", 3: "파일 다운로드", 4: "크롤링", 5: "수동 입력"}
PASSED, REVIEW = "형식 통과", "확인 필요"
SAME_AS_DATE = "same_as_date"
CHECKED_MARKS = {"o", "y", "yes", "✅", "확인"}
NULL_TOKENS = {"", "-", ".", "\u2014", "n/a", "na", "nan", "none", "null"}
KST = timezone(timedelta(hours=9))
USER_AGENT = "KFA-AI-agent-study/0.2 (+https://github.com/phm0305/KFA-ai-agent-study)"


# 실패(❌). 메시지가 조원 화면에 그대로 나가서 키·전체 주소를 넣지 않아요
class CollectError(Exception):
    """Shown to members verbatim, so it must never contain keys or full URLs."""


# 준비가 안 돼서 건너뜀(⏭). 키 없음, 설정 꺼짐, 사이트가 막음 등
class SkipStep(Exception):
    pass


# 지금 한국 시각 (수집 시각 칸에 씀)
def now_kst():
    return datetime.now(KST).strftime("%Y-%m-%d %H:%M")


# 빈칸인지 판단 (None, 빈 글자, NaN 모두 빈칸)
def is_blank(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


# '1,234' '(500)' '△500' '12.5%' → 숫자. 못 바꾸면 (None, 이유)
def to_number(value):
    if is_blank(value):
        return None, None
    if isinstance(value, bool):
        return None, f"숫자로 못 바꿈: '{value}'"
    if isinstance(value, numbers.Integral):
        return int(value), None
    if isinstance(value, numbers.Real):
        return (float(value), None) if math.isfinite(value) else (None, None)
    text = str(value).strip()
    if text.lower() in NULL_TOKENS:
        return None, None
    cleaned = text.replace(",", "").replace(" ", "").rstrip("%")
    negative = False
    if cleaned.startswith("(") and cleaned.endswith(")"):
        negative, cleaned = True, cleaned[1:-1]
    elif cleaned.startswith("△"):
        negative, cleaned = True, cleaned[1:]
    if re.fullmatch(r"[+-]?\d+", cleaned):
        number = int(cleaned)
    else:
        try:
            number = float(cleaned)
        except ValueError:
            return None, f"숫자로 못 바꿈: '{text}'"
        if not math.isfinite(number):
            return None, f"숫자로 못 바꿈: '{text}'"
    return (-number if negative else number), None


_DAY = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*[-./월]\s*(\d{1,2})\s*일?\.?")
_COMPACT_DAY = re.compile(r"(\d{4})(\d{2})(\d{2})")
_MONTH = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*월?\.?")
_COMPACT_MONTH = re.compile(r"(\d{4})(\d{2})")
_QUARTER = re.compile(r"(\d{4})\s*-?\s*Q([1-4])", re.IGNORECASE)
_YEAR = re.compile(r"(\d{4})\s*년?")
_TIME_SUFFIX = re.compile(r"[ T]\d{1,2}:\d{2}(:\d{2})?(\.\d+)?$")


# '2026.6.30.' '20260630' '2026년 6월' '2026Q2' 등 → 표준 날짜 글자. 못 바꾸면 (None, 이유)
def to_date(value):
    """Normalize to YYYY-MM-DD, YYYY-MM, YYYYQn or YYYY; anything else is a problem."""
    if is_blank(value):
        return None, None
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.strftime("%Y-%m-%d"), None
    text = _TIME_SUFFIX.sub("", str(value).strip())
    for pattern in (_DAY, _COMPACT_DAY):
        match = pattern.fullmatch(text)
        if match:
            try:
                return date(*map(int, match.groups())).isoformat(), None
            except ValueError:
                return None, f"없는 날짜: '{value}'"
    for pattern in (_MONTH, _COMPACT_MONTH):
        match = pattern.fullmatch(text)
        if match:
            year, month = map(int, match.groups())
            if 1 <= month <= 12:
                return f"{year:04d}-{month:02d}", None
            return None, f"없는 날짜: '{value}'"
    match = _QUARTER.fullmatch(text)
    if match:
        return f"{match.group(1)}Q{match.group(2)}", None
    match = _YEAR.fullmatch(text)
    if match:
        return match.group(1), None
    return None, f"날짜 모양을 모름: '{value}'"


# 이름에 그 글자가 들어간 첫 칸 찾기 (크롤링 표처럼 칸 이름이 조금씩 바뀔 때)
def find_col(table, keyword):
    for col in table.columns:
        if keyword in str(col):
            return col
    raise CollectError(f"'{keyword}'가 들어간 칸이 없어요. 지금 표의 칸: {[str(c) for c in table.columns]}")


# 넓은 표(칸마다 항목) → 긴 표(숫자 하나 = 한 줄). items·date_col을 여기서 씀
def long_from_table(table, items, date_col=None, keep=None):
    """Wide table (one column per item) -> one row per number.

    items: {raw column: (항목 이름, 단위)}. keep: {raw column: our column} copied per row.
    """
    if date_col is None:
        if isinstance(table.index, pd.RangeIndex):
            raise CollectError("날짜가 들어 있는 칸 이름을 date_col로 알려 주세요")
        table = table.reset_index()
        date_col = table.columns[0]
    keep = keep or {}
    missing = [c for c in [*items, date_col, *keep] if c not in table.columns]
    if missing:
        raise CollectError(f"표에 없는 칸: {missing}. 지금 표의 칸: {[str(c) for c in table.columns]}")
    parts = []
    for raw_col, (label, unit) in items.items():
        part = pd.DataFrame({
            "항목": label,
            "값": pd.Series(table[raw_col].tolist(), dtype=object),
            "단위": unit,
            "기준일": pd.Series(table[date_col].tolist(), dtype=object),
        })
        for raw, ours in keep.items():
            part[ours] = table[raw].tolist()
        parts.append(part)
    if not parts:
        raise CollectError("items가 비어 있어요. 어떤 칸을 숫자로 가져올지 적어 주세요")
    return pd.concat(parts, ignore_index=True)


# 꼬리표 채우기 + 숫자·날짜 정리 + 형식 통과/확인 필요 판정 (모든 단계 공통)
def finalize(table, *, level, source=None, source_url=None, reliability=None, basis=None,
             disclosed=None, base_date=None, stock_code=None, stock_name=None, unit=None, note=None):
    """Turn any table with at least 항목/값 into the shared record table.

    A cell already in the table wins over the keyword default. disclosed may be a
    date, NOT_APPLICABLE, or SAME_AS_DATE (copy 기준일, for prices and rates).
    """
    if level not in LEVELS:
        raise ValueError(f"level must be one of {list(LEVELS)}")
    defaults = {
        "종목코드": stock_code, "종목명": stock_name, "단위": unit, "기준": basis,
        "기준일": base_date, "출처": source, "출처 URL": source_url, "신뢰도": reliability,
    }
    collected_at = now_kst()
    method = f"{level} {LEVELS[level]}"
    out = []
    for raw in table.to_dict("records"):
        row = {col: raw.get(col) for col in COLUMNS}
        for col, default in defaults.items():
            if is_blank(row[col]) and default is not None:
                row[col] = default
        notes = [n for n in (raw.get("비고"), note) if not is_blank(n)]
        reasons = []

        if is_blank(row["공시일"]):
            if disclosed == SAME_AS_DATE:
                row["공시일"] = row["기준일"]
                notes.append("공시일은 기준일과 같다고 봄")
            elif disclosed is not None:
                row["공시일"] = disclosed
        row["수집 방법"] = method
        if is_blank(row["수집 시각"]):
            row["수집 시각"] = collected_at

        value, value_problem = to_number(row["값"])
        if value_problem:
            reasons.append(value_problem)
        else:
            row["값"] = value

        for col in ("기준일", "공시일"):
            if is_blank(row[col]) or row[col] == NOT_APPLICABLE:
                continue
            normalized, problem = to_date(row[col])
            if problem:
                reasons.append(f"{col} {problem}")
            else:
                row[col] = normalized

        for col in REQUIRED:
            if is_blank(row[col]) and not (col == "값" and value_problem):
                reasons.append(f"{col} 비어 있음")
        for col in COLUMNS:
            if row[col] == NOT_APPLICABLE and col not in NA_ALLOWED:
                reasons.append(f"{col}에는 '{NOT_APPLICABLE}'을 쓸 수 없음")

        base, disc = str(row["기준일"] or ""), str(row["공시일"] or "")
        if re.fullmatch(r"\d{4}-\d{2}(-\d{2})?", base) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", disc):
            if disc[:len(base)] < base:
                reasons.append("공시일이 기준일보다 빠름")
        if not is_blank(row["신뢰도"]) and row["신뢰도"] not in RELIABILITY:
            reasons.append("신뢰도는 상/중/하 중 하나")

        checked = str(raw.get("확인") or "").strip().lower() in CHECKED_MARKS
        if level == 4 and not checked:
            reasons.append("크롤링 값: 원문 화면과 한 번 대조")
        if level == 5 and not checked:
            reasons.append("손으로 옮긴 값: 원문과 대조")

        row["상태"] = REVIEW if reasons else PASSED
        row["비고"] = "; ".join([*dict.fromkeys(notes), *(f"⚠ {r}" for r in reasons)])
        out.append(row)

    records = pd.DataFrame(out, columns=COLUMNS)
    records["값"] = pd.Series([r["값"] for r in out], dtype=object)
    return records


# 모든 단계의 공통 출구: 받아온 표 → 공통 표 15칸
def collect_table(table, items=None, date_col=None, keep=None, **meta):
    """Single exit for every level: raw table in, record table out.

    Without items, the table must already have 항목/값 columns (typical for manual CSVs).
    """
    if not isinstance(table, pd.DataFrame):
        raise CollectError(f"표(DataFrame)가 아니라 {type(table).__name__}가 들어왔어요")
    if items is not None:
        table = long_from_table(table, items, date_col, keep)
    elif not {"항목", "값"} <= set(table.columns):
        raise CollectError(f"items 없이 쓰려면 '항목'과 '값' 칸이 있어야 해요. 지금 칸: {[str(c) for c in table.columns]}")
    return finalize(table, **meta)


# 확인 필요 줄의 ⚠ 이유를 종류별로 세기
def review_reasons(records):
    reasons = Counter()
    for note in records.loc[records["상태"] == REVIEW, "비고"].fillna(""):
        for part in note.split("; "):
            if part.startswith("⚠ "):
                reasons[re.sub(r"'.*'", "'…'", part[2:])] += 1
    return reasons


# 공통 표 모양 검사. 맞으면 '✅ 칸 모양 OK' 요약을 출력
def check_records(records, show=True):
    problems = []
    if not isinstance(records, pd.DataFrame):
        problems.append(f"표(DataFrame)가 아니에요: {type(records).__name__}")
    elif list(records.columns) != COLUMNS:
        missing = [c for c in COLUMNS if c not in records.columns]
        extra = [c for c in records.columns if c not in COLUMNS]
        problems.append(f"칸 이름·순서가 달라요 (빠진 칸 {missing}, 남는 칸 {extra})")
    else:
        bad_state = ~records["상태"].isin([PASSED, REVIEW])
        if bad_state.any():
            problems.append(f"상태 칸에 '{PASSED}'/'{REVIEW}' 말고 다른 값이 {int(bad_state.sum())}줄")
        blank = records[REQUIRED].apply(lambda col: col.map(is_blank)).any(axis=1)
        hidden = blank & (records["상태"] == PASSED)
        if hidden.any():
            problems.append(f"필수 칸이 비었는데 '{PASSED}'인 줄이 {int(hidden.sum())}개 (finalize를 거치지 않은 듯)")
    if show:
        if problems:
            for p in problems:
                print("❌", p)
        else:
            passed = int((records["상태"] == PASSED).sum())
            print(f"✅ 칸 모양 OK: {len(records)}줄 (형식 통과 {passed} · 확인 필요 {len(records) - passed})")
            for reason, count in review_reasons(records).most_common(5):
                print(f"   ⚠ {reason}: {count}줄")
    return not problems


# CSV 저장: 전부(stem.csv) + 확인 필요만(stem_확인필요.csv). 엑셀에서 안 깨지게 utf-8-sig
def save_records(records, stem="records"):
    all_path, review_path = f"{stem}.csv", f"{stem}_확인필요.csv"
    records.to_csv(all_path, index=False, encoding="utf-8-sig")
    records[records["상태"] == REVIEW].to_csv(review_path, index=False, encoding="utf-8-sig")
    return all_path, review_path


STEP_RESULTS = []


# 예시 하나 실행 → 요약표에 ✅ ⏭ ❌ 한 줄. 어떤 오류가 나도 '모두 실행'을 멈추지 않음
def run_step(label, level, func):
    """Run one source without ever stopping 'Run all'; errors become a summary row."""
    entry = {"단계": f"{level} {LEVELS[level]}", "예시": label, "결과": "❌", "줄 수": 0, "확인 필요": 0, "메모": ""}
    STEP_RESULTS[:] = [e for e in STEP_RESULTS if e["예시"] != label]
    STEP_RESULTS.append(entry)
    try:
        records = func()
    except SkipStep as e:
        entry.update(결과="⏭", 메모=str(e))
        print(f"⏭ {label}: {e}")
        return None
    except CollectError as e:
        entry["메모"] = str(e)
        print(f"❌ {label}: {e}")
        return None
    except requests.RequestException as e:
        entry["메모"] = f"인터넷 연결 문제로 못 받아왔어요 ({type(e).__name__}). 잠시 뒤 이 셀만 다시 실행해 주세요"
        print(f"❌ {label}: {entry['메모']}")
        return None
    except Exception as e:
        entry["메모"] = f"예상 못 한 오류 ({type(e).__name__}). 셀을 다시 실행해 보고, 그래도 안 되면 조 톡방에 공유"
        print(f"❌ {label}: {entry['메모']}")
        return None
    if not check_records(records, show=False):
        entry["메모"] = "공통 표 모양이 아니에요. check_records(결과)로 무엇이 다른지 확인"
        check_records(records)
        return None
    review = int((records["상태"] == REVIEW).sum())
    entry.update(결과="✅", **{"줄 수": len(records), "확인 필요": review})
    print(f"✅ {label}: {len(records)}줄 (확인 필요 {review}줄)")
    return records


# 지금까지 실행한 예시의 요약표
def step_summary():
    return pd.DataFrame(STEP_RESULTS, columns=["단계", "예시", "결과", "줄 수", "확인 필요", "메모"])


# 주소에 요청 보내기. 오류 메시지에는 사이트 이름만 넣어 키가 화면에 안 나오게 함
def http_get(url, params=None, timeout=30):
    host = urlparse(url).netloc
    try:
        resp = requests.get(url, params=params, timeout=timeout, headers={"User-Agent": USER_AGENT})
    except requests.Timeout:
        raise CollectError(f"{host} 응답이 {timeout}초 넘게 없어요. 잠시 뒤 다시 실행해 주세요") from None
    except requests.RequestException:
        raise CollectError(f"{host}에 연결하지 못했어요. 인터넷이나 사이트 상태를 확인해 주세요") from None
    if resp.status_code != 200:
        hint = {401: "키나 로그인이 필요해요", 403: "사이트가 접근을 막았어요", 404: "주소가 바뀌었을 수 있어요",
                429: "요청이 너무 잦아요. 1분 뒤 다시"}.get(resp.status_code, "")
        raise CollectError(f"{host}가 {resp.status_code} 응답을 줬어요. {hint}".strip())
    return resp


# [1단계] JSON 응답에서 'a.b.c' 위치의 목록을 표로. 위치가 틀리면 그 자리에 뭐가 있는지 알려줌
def read_json_table(url, rows_path, params=None):
    """Fetch JSON and return the list found at a dotted path (e.g. 'StatisticSearch.row') as a table."""
    try:
        data = http_get(url, params).json()
    except ValueError:
        raise CollectError("응답이 JSON이 아니에요. 주소의 형식(json/xml) 부분을 확인해 주세요") from None
    current = data
    for key in rows_path.split("."):
        if isinstance(current, dict) and key in current:
            current = current[key]
            continue
        here = list(current) if isinstance(current, dict) else type(current).__name__
        preview = json.dumps(data, ensure_ascii=False)[:200]
        raise CollectError(f"응답에서 '{key}'를 못 찾았어요. 그 자리에 있는 것: {here} / 응답 앞부분: {preview}")
    rows = [current] if isinstance(current, dict) else current
    if not isinstance(rows, list) or not rows:
        raise CollectError(f"'{rows_path}' 아래가 비어 있어요")
    return pd.json_normalize(rows)


DART_URL = "https://opendart.fss.or.kr/api"
DART_REPORT = {1: "11013", 2: "11012", 3: "11014", 4: "11011"}
QUARTER_END = {1: "03-31", 2: "06-30", 3: "09-30", 4: "12-31"}
DART_ERRORS = {
    "010": "등록되지 않은 키예요. Secrets의 키 값을 확인해 주세요",
    "011": "사용할 수 없는 키예요",
    "013": "조회된 데이터가 없어요. 아직 공시 전인 분기인지 확인해 주세요",
    "020": "오늘 요청 한도를 넘었어요. 내일 다시 해 주세요",
    "100": "요청 값이 잘못됐어요 (연도·분기·연결/별도)",
    "800": "DART 시스템 점검 중이에요",
}
DART_ITEMS = {
    "ifrs-full_Revenue": ("매출액", True),
    "dart_OperatingIncomeLoss": ("영업이익", True),
    "ifrs-full_ProfitLoss": ("당기순이익", True),
    "ifrs-full_Assets": ("자산총계", False),
    "ifrs-full_Liabilities": ("부채총계", False),
    "ifrs-full_Equity": ("자본총계", False),
}
_DART_CORPS = {}


def _dart_status(data):
    status = data.get("status")
    if status != "000":
        raise CollectError(f"DART {status}: {DART_ERRORS.get(status, data.get('message', '알 수 없는 응답'))}")


# [1단계] 종목코드 → DART 회사 번호 (번호표는 한 번만 받아서 기억)
def dart_corp(stock_code, api_key):
    if not _DART_CORPS:
        resp = http_get(f"{DART_URL}/corpCode.xml", {"crtfc_key": api_key})
        try:
            xml_bytes = zipfile.ZipFile(io.BytesIO(resp.content)).read("CORPCODE.xml")
        except zipfile.BadZipFile:
            try:
                _dart_status(resp.json())
            except ValueError:
                pass
            raise CollectError("DART 회사 번호표를 받지 못했어요. 키를 확인해 주세요") from None
        for item in ET.fromstring(xml_bytes).iter("list"):
            code = (item.findtext("stock_code") or "").strip()
            if code:
                _DART_CORPS[code] = (item.findtext("corp_code"), item.findtext("corp_name"))
    if stock_code not in _DART_CORPS:
        raise CollectError(f"{stock_code}: 상장사 목록에 없어요. 종목코드 6자리를 확인해 주세요")
    return _DART_CORPS[stock_code]


# [1단계] DART 재무제표 응답에서 6개 계정만 골라 3개월·누적을 나눠 적기
def dart_rows_to_long(rows, year, quarter):
    """Pick six accounts by standard account_id; flows get 3-month and cumulative rows."""
    base_note = "기준일은 12월 결산 가정"
    out, seen = [], set()
    for row in rows:
        account = row.get("account_id")
        if account not in DART_ITEMS or account in seen:
            continue
        name, is_flow = DART_ITEMS[account]
        if is_flow and row.get("sj_div") not in ("IS", "CIS"):
            continue
        seen.add(account)
        rcept_no = row.get("rcept_no", "")
        common = {
            "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
            "공시일": rcept_no[:8], "출처 URL": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
        }
        if not is_flow:
            parts = [(name, row.get("thstrm_amount"), base_note)]
        elif quarter == 4:
            parts = [(f"{name}(연간)", row.get("thstrm_amount"), base_note + "; 4분기 3개월치 = 연간 - 3분기 누적")]
        elif quarter == 1:
            parts = [(f"{name}(3개월)", row.get("thstrm_amount"), base_note + "; 1분기는 3개월 = 누적")]
        else:
            parts = [(f"{name}(3개월)", row.get("thstrm_amount"), base_note),
                     (f"{name}(누적)", row.get("thstrm_add_amount"), base_note)]
        for label, amount, memo in parts:
            out.append({**common, "항목": label, "값": amount, "비고": memo})
    for account, (name, _) in DART_ITEMS.items():
        if account not in seen:
            out.append({"항목": name, "값": None, "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
                        "비고": f"이 회사 공시에서 표준 계정({account})을 못 찾음"})
    return pd.DataFrame(out)


# [1단계] DART 재무제표 → 공통 표 (매출액·영업이익·당기순이익·자산·부채·자본)
def collect_dart(stock_code, year, quarter, fs_div, api_key):
    if not api_key:
        raise SkipStep("DART 키가 없어요 (Secrets에 DART_API_KEY 등록 후 다시)")
    if quarter not in DART_REPORT or fs_div not in ("CFS", "OFS"):
        raise CollectError("QUARTER는 1~4, FS_DIV는 'CFS'(연결) 또는 'OFS'(별도)")
    corp_code, corp_name = dart_corp(stock_code, api_key)
    data = http_get(f"{DART_URL}/fnlttSinglAcntAll.json", {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": str(year),
        "reprt_code": DART_REPORT[quarter], "fs_div": fs_div,
    }).json()
    _dart_status(data)
    return collect_table(
        dart_rows_to_long(data["list"], year, quarter),
        level=1, source="DART 단일회사 전체 재무제표 API", reliability="상",
        basis="연결" if fs_div == "CFS" else "별도", stock_code=stock_code, stock_name=corp_name,
        source_url="https://opendart.fss.or.kr",
    )


# [1단계] 한국은행 ECOS 통계 1개 → 공통 표. 키가 없으면 ⏭
def collect_ecos(api_key, stat_code, cycle, start, end, item_code, label, unit):
    if not api_key:
        raise SkipStep("ECOS 키가 없어요 (선택 사항: ecos.bok.or.kr에서 무료 발급 후 Secrets에 ECOS_API_KEY)")
    url = f"https://ecos.bok.or.kr/api/StatisticSearch/{api_key}/json/kr/1/1000/{stat_code}/{cycle}/{start}/{end}/{item_code}"
    try:
        data = http_get(url).json()
    except ValueError:
        raise CollectError("ECOS 응답이 JSON이 아니에요") from None
    if "RESULT" in data:
        raise CollectError(f"ECOS: {data['RESULT'].get('MESSAGE', data['RESULT'])}")
    table = pd.json_normalize(data["StatisticSearch"]["row"])
    return collect_table(
        table, {"DATA_VALUE": (label, unit)}, date_col="TIME",
        level=1, source="한국은행 ECOS Open API", reliability="상", basis=NOT_APPLICABLE,
        source_url=f"https://ecos.bok.or.kr/#/SearchStat?statCode={stat_code}",
    )


# [3·5단계] 주소나 파일 경로의 CSV·엑셀 읽기 (한글 깨짐 cp949, 이름만 .xls인 파일도 처리)
# 칸 이름과 글자 값의 앞뒤 공백은 지워요 (화면에서 안 보이는 차이라서)
def read_table(source, sheet_name=0, skiprows=None):
    """CSV/Excel from a URL or a path. Handles cp949 CSVs and HTML disguised as .xls.

    Leading/trailing spaces in column names and text cells are removed: they are
    invisible on screen, so no reader can mean anything by them.
    """
    table = _read_table_raw(source, sheet_name, skiprows)
    table.columns = [c.strip() if isinstance(c, str) else c for c in table.columns]
    for col in table.columns:
        if not pd.api.types.is_numeric_dtype(table[col]):
            table[col] = table[col].map(lambda v: v.strip() if isinstance(v, str) else v)
    return table


def _read_table_raw(source, sheet_name, skiprows):
    if str(source).startswith(("http://", "https://")):
        content, name = http_get(str(source)).content, urlparse(str(source)).path.lower()
    else:
        path = Path(source)
        if not path.exists():
            raise CollectError(f"파일이 없어요: {path.name}. 경로와 파일 이름(띄어쓰기·확장자)을 확인해 주세요")
        content, name = path.read_bytes(), path.name.lower()
    if content.lstrip()[:1] == b"<":
        try:
            return pd.read_html(io.StringIO(content.decode("utf-8", errors="replace")))[0]
        except ValueError:
            raise CollectError("파일 안이 표가 아닌 웹페이지예요. 받은 파일을 열어 확인해 주세요") from None
    if name.endswith((".xlsx", ".xlsm", ".xls")):
        try:
            return pd.read_excel(io.BytesIO(content), sheet_name=sheet_name, skiprows=skiprows)
        except ImportError:
            raise CollectError("이 엑셀 형식을 읽는 도구가 없어요. 엑셀에서 .xlsx로 다시 저장해 주세요") from None
    for encoding in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(io.BytesIO(content), encoding=encoding, skiprows=skiprows)
        except UnicodeDecodeError:
            continue
    raise CollectError("CSV 글자를 못 읽었어요 (utf-8, cp949 둘 다 실패). 엑셀에서 'CSV UTF-8'로 다시 저장해 주세요")


# [4단계] 사이트의 robots.txt가 코드 수집을 허락하는지 (모르면 None)
def robots_allowed(url):
    """True/False from robots.txt; None when it cannot be read (caller decides)."""
    parts = urlparse(url)
    try:
        resp = requests.get(f"{parts.scheme}://{parts.netloc}/robots.txt", timeout=15,
                            headers={"User-Agent": USER_AGENT})
    except requests.RequestException:
        return None
    if resp.status_code in (401, 403):
        return False
    if resp.status_code >= 400:
        return True
    parser = urllib.robotparser.RobotFileParser()
    parser.parse(resp.text.splitlines())
    return parser.can_fetch(USER_AGENT, url)


# [4단계] 웹페이지 화면의 표 읽기. robots.txt가 막으면 ⏭
def read_html_table(url, match=".+", index=0):
    host = urlparse(url).netloc
    allowed = robots_allowed(url)
    if allowed is False:
        raise SkipStep(f"{host}의 robots.txt가 코드 수집을 막고 있어요. 이 자료는 5단계(수동)로 옮겨 주세요")
    if allowed is None:
        print(f"⚠ {host}의 robots.txt를 확인하지 못했어요. 사이트 이용약관을 직접 확인해 주세요")
    resp = http_get(url)
    if resp.encoding is None or resp.encoding.lower() == "iso-8859-1":
        resp.encoding = resp.apparent_encoding
    try:
        tables = pd.read_html(io.StringIO(resp.text), match=match)
    except ValueError:
        raise CollectError(f"페이지에서 '{match}'가 들어간 표를 못 찾았어요. 표가 자바스크립트로 그려지는 페이지일 수 있어요") from None
    if index >= len(tables):
        raise CollectError(f"표가 {len(tables)}개뿐이에요. index를 0~{len(tables) - 1} 중에서 골라 주세요")
    table = tables[index]
    if isinstance(table.columns, pd.MultiIndex):
        table.columns = [" ".join(dict.fromkeys(str(c) for c in col)) for col in table.columns]
    return table


DRIVE_ROOT = "/content/drive/MyDrive/KFA_자료"
CARD_COLUMNS = ["파일명", "출처", "URL", "기준일", "신뢰도"]


# [5단계] 구글 드라이브 연결 후 KFA_자료 폴더 위치 돌려주기 (Colab 전용)
def mount_drive():
    try:
        from google.colab import drive
    except ImportError:
        raise CollectError("구글 드라이브 연결은 Colab에서만 돼요") from None
    drive.mount("/content/drive")
    if not Path(DRIVE_ROOT).exists():
        raise CollectError("내 드라이브 맨 위에 'KFA_자료' 폴더가 없어요 (이름의 밑줄까지 똑같이)")
    return Path(DRIVE_ROOT)


# [5단계] 출처카드 읽기: CSV 경로 또는 구글 시트 제목
def read_source_cards(source="출처카드"):
    """A CSV path, or (in Colab) the title of a Google Sheet."""
    if str(source).lower().endswith(".csv") or Path(str(source)).exists():
        return read_table(source)
    try:
        from google.colab import auth
    except ImportError:
        raise CollectError("구글 시트는 Colab에서만 읽을 수 있어요. 여기서는 CSV 경로를 주세요") from None
    import gspread
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    client = gspread.authorize(creds)
    try:
        sheet = client.open(source).sheet1
    except gspread.SpreadsheetNotFound:
        files = client.list_spreadsheet_files()
        title = pick_sheet_title(source, [f["name"] for f in files])
        print(f"'{title}'를 {source}로 읽었어요")
        sheet = client.open_by_key(next(f["id"] for f in files if f["name"] == title)).sheet1
    return pd.DataFrame(sheet.get_all_records())


# [5단계] 시트 이름이 정확히 같지 않을 때 띄어쓰기만 다른 시트 찾기
def pick_sheet_title(wanted, titles):
    """The one title equal to wanted once all whitespace is removed; stops on zero or several."""
    squeeze = lambda s: re.sub(r"\s+", "", str(s))
    matches = [t for t in titles if squeeze(t) == squeeze(wanted)]
    if not matches:
        raise CollectError(f"'{wanted}'라는 구글 시트를 못 찾았어요. 시트 이름을 확인해 주세요")
    if len(matches) > 1:
        raise CollectError(f"'{wanted}'와 띄어쓰기만 다른 시트가 {len(matches)}개예요: {matches}. "
                           f"쓸 시트 하나만 이름을 '{wanted}'로 바꿔 주세요")
    return matches[0]


# [3·5단계] 출처카드에 적힌 파일만 읽어서 공통 표로. 못 읽는 파일은 이유와 함께 건너뜀
def collect_from_cards(cards, files_dir):
    """Read files listed on the source cards. Only CSV/XLSX already shaped as 항목/값/단위/기준일.

    An optional 공시일 card column fills 공시일 the same way 기준일 does: a cell in the file wins.
    """
    missing = [c for c in CARD_COLUMNS if c not in cards.columns]
    if missing:
        raise CollectError(f"출처카드에 없는 칸: {missing}")
    results, skipped = [], []
    for card in cards.to_dict("records"):
        name = str(card["파일명"]).strip()
        path = Path(files_dir) / name
        if not path.exists():
            skipped.append((name, "files/ 폴더에 이 이름의 파일이 없어요"))
            continue
        if path.suffix.lower() not in (".csv", ".xlsx"):
            skipped.append((name, "PDF·글 자료는 4회차에 AI가 읽어요"))
            continue
        table = read_table(path)
        if not {"항목", "값"} <= set(table.columns):
            skipped.append((name, "'항목·값·단위·기준일' 칸 모양이 아니에요. 3단계 예시처럼 items로 바꿔 주세요. "
                                  f"지금 칸: {[str(col) for col in table.columns]}"))
            continue
        level_match = re.match(r"\s*([1-5])", str(card.get("수집 방법") or ""))
        disclosed = None if is_blank(card.get("공시일")) else card["공시일"]
        results.append(collect_table(
            table, level=int(level_match.group(1)) if level_match else 5, source=card["출처"], source_url=card["URL"],
            reliability=card["신뢰도"], base_date=card["기준일"], disclosed=disclosed, note=f"출처카드: {name}",
        ))
    for name, why in skipped:
        print(f"   ⏭ {name}: {why}")
    if not results:
        raise SkipStep(f"출처카드 {len(cards)}줄 중 읽을 수 있는 숫자 파일이 없어요")
    return pd.concat(results, ignore_index=True)


# 키 읽기: Colab이면 Secrets, 내 컴퓨터면 .env. 값은 절대 출력하지 않음
def get_key(name):
    """Colab Secrets first, then .env / environment. Never print the value."""
    try:
        from google.colab import userdata
    except ImportError:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            pass
        return os.getenv(name) or None
    try:
        return userdata.get(name) or None
    except Exception:
        return None

STEP_RESULTS.clear()
print("✅ 공통 도구 준비 완료")

✅ 공통 도구 준비 완료


In [2]:
# 내 소스: DART 2026년 반기 연결 영업이익
def my_records():
    api_key = get_key("DART_API_KEY")
    if not api_key:
        raise SkipStep("왼쪽 열쇠에서 DART_API_KEY의 노트북 액세스를 켜 주세요")

    companies = [
        ("016360", "삼성증권", "00104856"),
        ("039490", "키움증권", "00296290"),
    ]
    tables = []

    for code, name, corp_code in companies:
        data = http_get(
            f"{DART_URL}/fnlttSinglAcntAll.json",
            params={
                "crtfc_key": api_key,
                "corp_code": corp_code,
                "bsns_year": "2026",
                "reprt_code": "11012",
                "fs_div": "CFS",
            },
        ).json()
        _dart_status(data)

        # 손익계산서에서 영업이익 찾기
        candidates = [
            row for row in data.get("list", [])
            if row.get("sj_div") in ("IS", "CIS")
            and (
                row.get("account_id") == "dart_OperatingIncomeLoss"
                or row.get("account_nm", "").strip() == "영업이익"
            )
        ]

        if not candidates:
            raise CollectError(f"{name}: 영업이익 항목을 찾지 못했어요")

        # 일반 손익계산서(IS)를 우선 사용
        candidates.sort(key=lambda row: row.get("sj_div") != "IS")
        row = candidates[0]
        receipt = row.get("rcept_no", "")

        raw = pd.DataFrame([
            {
                "항목": "영업이익(2분기 3개월)",
                "값": row.get("thstrm_amount"),
            },
            {
                "항목": "영업이익(상반기 누적)",
                "값": row.get("thstrm_add_amount"),
            },
        ])

        tables.append(collect_table(
            raw,
            level=1,
            source="DART 단일회사 전체 재무제표 API",
            source_url=(
                f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={receipt}"
                if receipt else None
            ),
            reliability="상",
            basis="연결",
            base_date="2026-06-30",
            disclosed=receipt[:8] or None,
            stock_code=code,
            stock_name=name,
            unit="원",
            note="2026년 반기보고서. 2분기 3개월과 상반기 누적을 구분",
        ))

    return pd.concat(tables, ignore_index=True)


records = run_step("내 소스: 증권사 연결 영업이익", 1, my_records)

if records is not None:
    display(records)
    check_records(records)
    paths = save_records(records, "w02_records")
    print("💾 저장:", *paths)

✅ 내 소스: 증권사 연결 영업이익: 4줄 (확인 필요 0줄)


,종목코드,종목명,항목,값,단위,기준,기준일,공시일,출처,출처 URL,신뢰도,수집 방법,수집 시각,상태,비고
0,016360,삼성증권,영업이익(2분기 3개월),675794814040,원,연결,2026-06-30,2026-08-14,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,상,1 공식 API,2026-11-04 15:47,형식 통과,2026년 반기보고서. 2분기 3개월과 상반기 누적을 구분
1,016360,삼성증권,영업이익(상반기 누적),1285326557425,원,연결,2026-06-30,2026-08-14,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,상,1 공식 API,2026-11-04 15:47,형식 통과,2026년 반기보고서. 2분기 3개월과 상반기 누적을 구분
2,039490,키움증권,영업이익(2분기 3개월),788949832477,원,연결,2026-06-30,2026-08-13,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,상,1 공식 API,2026-11-04 15:48,형식 통과,2026년 반기보고서. 2분기 3개월과 상반기 누적을 구분
3,039490,키움증권,영업이익(상반기 누적),1410170449849,원,연결,2026-06-30,2026-08-13,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,상,1 공식 API,2026-11-04 15:48,형식 통과,2026년 반기보고서. 2분기 3개월과 상반기 누적을 구분


✅ 칸 모양 OK: 4줄 (형식 통과 4 · 확인 필요 0)
💾 저장: w02_records.csv w02_records_확인필요.csv
